# 🎭 OpenAI Multi-Agent Conversation & Stand-up Comedy

This notebook demonstrates the **Multi-Agent** design pattern using OpenAI's API where multiple AI agents collaborate — or argue — to perform stand-up comedy routines, critique each other, and refine jokes iteratively.

## Concept Overview

| Agent | Role |
|---|---|
| `ComedianAgent` | Writes and delivers stand-up jokes |
| `CriticAgent` | Reviews jokes for timing, originality, punchline strength |
| `HecklerAgent` | Interrupts with snarky remarks (adds chaos!) |
| `MCAgent` | Orchestrates the show and summarises the performance |

---

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install openai --quiet

## 🔑 Step 2: Setup API Key

In [ ]:
import os
from openai import OpenAI

# Set your OpenAI API key here or use environment variable
os.environ["OPENAI_API_KEY"] = "your-api-key-here"  # Replace with your key

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-4o"

print("✅ OpenAI client initialized!")

## 🤖 Step 3: Define Agent Personas

Each agent has a distinct **system prompt** that defines its personality, style, and objectives.

In [ ]:
# ─── Agent System Prompts ───────────────────────────────────────────────────

COMEDIAN_SYSTEM = """
You are a stand-up comedian named 'Chuckles McGee'. 
Your comedy style is observational, self-deprecating, and slightly absurdist.
When asked to perform, write 2-3 jokes on the given topic.
Format each joke with a setup and a clear punchline.
Keep each joke under 4 sentences. Be funny, original, and punchy.
"""

CRITIC_SYSTEM = """
You are a sharp comedy critic named 'Reginald Highbrow'.
You evaluate stand-up jokes on three dimensions:
  1. Originality (1-10)
  2. Timing/Structure (1-10) 
  3. Punchline Strength (1-10)
Give an overall score and ONE specific suggestion to improve the weakest joke.
Be constructive but brutally honest. Use formal language.
"""

HECKLER_SYSTEM = """
You are an enthusiastic audience heckler named 'Big Mouth Barry'.
You interrupt comedy sets with:
  - Groan-worthy puns related to the jokes
  - Loud unsolicited opinions
  - One genuinely funny alternative punchline
Keep your heckling to 2-3 short outbursts. Be chaotic but entertaining.
"""

MC_SYSTEM = """
You are the Master of Ceremonies (MC) named 'Smooth Operator Sam'.
Your job is to:
  1. Introduce the comedy show and set the scene
  2. Transition between different parts of the show
  3. Summarize the evening's performance with a final verdict
Be warm, witty, and keep the energy high. Use short punchy sentences.
"""

print("✅ Agent personas defined!")

## 🛠️ Step 4: Build the Agent Class

A reusable `Agent` class that wraps OpenAI completions and maintains **conversation history** for multi-turn dialogue.

In [ ]:
from typing import Optional
import textwrap

class Agent:
    """A conversational agent with persistent memory and a defined persona."""

    def __init__(self, name: str, system_prompt: str, model: str = MODEL):
        self.name = name
        self.model = model
        self.system_prompt = system_prompt
        self.history: list[dict] = []  # Conversation memory

    def chat(self, user_message: str, temperature: float = 0.85) -> str:
        """Send a message and get a response. History is maintained."""
        self.history.append({"role": "user", "content": user_message})

        response = client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.system_prompt},
                *self.history
            ],
            temperature=temperature,
        )

        reply = response.choices[0].message.content.strip()
        self.history.append({"role": "assistant", "content": reply})
        return reply

    def reset(self):
        """Clear conversation history."""
        self.history = []

    def display(self, message: str):
        """Pretty-print the agent's message."""
        separator = "─" * 60
        print(f"\n{'🎤' if 'Comedian' in self.name else '🎩' if 'MC' in self.name else '🧐' if 'Critic' in self.name else '📢'} [{self.name}]")
        print(separator)
        print(textwrap.fill(message, width=70))
        print(separator)


# ─── Instantiate all agents ──────────────────────────────────────────────────
comedian = Agent("Comedian - Chuckles McGee", COMEDIAN_SYSTEM)
critic   = Agent("Critic - Reginald Highbrow", CRITIC_SYSTEM)
heckler  = Agent("Heckler - Big Mouth Barry", HECKLER_SYSTEM)
mc       = Agent("MC - Smooth Operator Sam", MC_SYSTEM)

print("✅ All agents instantiated and ready!")

## 🎪 Step 5: Run the Comedy Show!

The **orchestration loop** manages turn-taking between agents. This is the core of the multi-agent pattern.

In [ ]:
def run_comedy_show(topic: str):
    """
    Orchestrates a full multi-agent comedy show on a given topic.
    
    Flow:
      MC intro → Comedian performs → Heckler interrupts 
      → Critic reviews → Comedian revises → MC closes
    """
    print("\n" + "═" * 65)
    print(f"  🎭  TONIGHT'S TOPIC: '{topic.upper()}'")
    print("═" * 65)

    # ── Turn 1: MC opens the show ──────────────────────────────────────────
    intro = mc.chat(f"Open the comedy show. Tonight's topic is: '{topic}'")
    mc.display(intro)

    # ── Turn 2: Comedian performs ──────────────────────────────────────────
    performance = comedian.chat(
        f"Perform your stand-up routine on the topic: '{topic}'. "
        f"Deliver 2-3 original jokes with clear setups and punchlines."
    )
    comedian.display(performance)

    # ── Turn 3: Heckler interrupts ─────────────────────────────────────────
    heckling = heckler.chat(
        f"You just heard these jokes about '{topic}':\n\n{performance}\n\n"
        f"Heckle the comedian with your outbursts!"
    )
    heckler.display(heckling)

    # ── Turn 4: Critic reviews ─────────────────────────────────────────────
    critique = critic.chat(
        f"Please critique these stand-up jokes about '{topic}':\n\n{performance}\n\n"
        f"Score each dimension and suggest one improvement."
    )
    critic.display(critique)

    # ── Turn 5: Comedian responds to critique ──────────────────────────────
    revised = comedian.chat(
        f"The critic said:\n\n{critique}\n\n"
        f"Revise your weakest joke based on this feedback. "
        f"Show the original and the improved version."
    )
    comedian.display(revised)

    # ── Turn 6: MC closes the show ─────────────────────────────────────────
    closing = mc.chat(
        f"Close the comedy show. Briefly summarize what happened: "
        f"comedian performed, got heckled, critic scored them, and they revised. "
        f"Give a fun final verdict on the evening."
    )
    mc.display(closing)

    print("\n" + "═" * 65)
    print("  🎉  SHOW'S OVER — THANKS FOR LAUGHING!")
    print("═" * 65)


# ── Run the show! ────────────────────────────────────────────────────────────
run_comedy_show(topic="Artificial Intelligence")

## 🔄 Step 6: Agent-to-Agent Debate — Who's Funnier?

A **debate loop** where two comedian agents argue about comedy styles, with the critic as judge.

In [ ]:
def comedy_debate(topic: str, rounds: int = 2):
    """
    Two comedians debate their joke approaches.
    The critic picks a winner each round.
    """
    # Create two distinct comedians
    comedian_a = Agent("Comedian A - DarkWit Danny", 
        "You are a dark-humor comedian. Your jokes are dry, edgy, and unexpected. "
        "Write ONE sharp joke per turn. Be provocative but not offensive.")
    
    comedian_b = Agent("Comedian B - Sunny Side Sue",
        "You are a wholesome, feel-good comedian. Your jokes are warm, relatable, "
        "and family-friendly. Write ONE heartwarming joke per turn. Be uplifting.")

    round_judge = Agent("Judge - Impartial Ivan",
        "You are an impartial comedy judge. Given two jokes on the same topic, "
        "pick the funnier one with a brief reason. Be fair and decisive.")

    print("\n" + "⚔️ " * 20)
    print(f"  COMEDY DEBATE: '{topic.upper()}' — {rounds} ROUNDS")
    print("⚔️ " * 20)

    scores = {"Comedian A": 0, "Comedian B": 0}

    for r in range(1, rounds + 1):
        print(f"\n📍 ROUND {r}")
        print("-" * 40)

        joke_a = comedian_a.chat(f"Tell one joke about '{topic}'.")
        comedian_a.display(joke_a)

        joke_b = comedian_b.chat(f"Tell one joke about '{topic}'.")
        comedian_b.display(joke_b)

        verdict = round_judge.chat(
            f"Topic: '{topic}'\n\n"
            f"Joke A (Dark Humor):\n{joke_a}\n\n"
            f"Joke B (Wholesome):\n{joke_b}\n\n"
            f"Who wins this round? Explain in 2 sentences."
        )
        round_judge.display(verdict)

        # Tally scores based on verdict
        if "Joke A" in verdict or "Dark" in verdict or "Comedian A" in verdict:
            scores["Comedian A"] += 1
            print("  🏆 Point to: Comedian A!")
        elif "Joke B" in verdict or "Wholesome" in verdict or "Comedian B" in verdict:
            scores["Comedian B"] += 1
            print("  🏆 Point to: Comedian B!")
        else:
            print("  🤝 Draw!")

    # Final result
    print(f"\n{'=' * 40}")
    print(f"  FINAL SCORES:")
    print(f"  Comedian A (Dark): {scores['Comedian A']}")
    print(f"  Comedian B (Wholesome): {scores['Comedian B']}")
    winner = max(scores, key=scores.get)
    print(f"  🥇 WINNER: {winner}!")
    print(f"{'=' * 40}")


comedy_debate(topic="Monday Mornings", rounds=2)

## 🧠 Step 7: Collaborative Joke Writing

Agents **collaborate** to build a single joke together — setup from one, punchline from another, then polished by a third.

In [ ]:
def collaborative_joke(topic: str):
    """
    Three agents co-author a single joke:
    1. Setup Writer → creates the setup
    2. Punchline Specialist → delivers the punchline
    3. Polisher → refines the complete joke
    """
    setup_agent = Agent("Setup Writer",
        "You write ONLY joke setups — the first part that creates expectation. "
        "One sentence max. Do NOT include a punchline. Leave it hanging.")
    
    punch_agent = Agent("Punchline Specialist",
        "Given a joke setup, you write ONLY the punchline. "
        "Make it subvert expectations cleverly. One sentence max.")
    
    polish_agent = Agent("Joke Polisher",
        "Given a setup and punchline, you refine the complete joke for "
        "better flow, word choice, and comedic timing. Output the final joke only.")

    print(f"\n🔨 COLLABORATIVE JOKE BUILDING — Topic: '{topic}'")
    print("-" * 50)

    # Step 1: Generate setup
    setup = setup_agent.chat(f"Write a joke setup about: '{topic}'")
    print(f"\n✏️  [Setup Writer]\n   Setup: {setup}")

    # Step 2: Generate punchline
    punchline = punch_agent.chat(f"Here is the setup: '{setup}'\nWrite the punchline.")
    print(f"\n💥 [Punchline Specialist]\n   Punchline: {punchline}")

    # Step 3: Polish the joke
    final_joke = polish_agent.chat(
        f"Setup: {setup}\nPunchline: {punchline}\n\n"
        f"Polish and present the complete joke."
    )
    print(f"\n✨ [Polished Final Joke]")
    print("─" * 50)
    print(final_joke)
    print("─" * 50)


collaborative_joke("ChatGPT replacing jobs")

## 📊 Step 8: Multi-Topic Batch Performance

Run the comedian across **multiple topics** and have the critic score each. Visualize performance.

In [ ]:
import json
import re

def batch_comedy_evaluation(topics: list[str]) -> list[dict]:
    """
    Run the comedian + critic pipeline across multiple topics.
    Returns structured scoring data.
    """
    # Comedian with JSON-structured critic
    batch_comedian = Agent("Batch Comedian", COMEDIAN_SYSTEM)
    batch_critic = Agent("Batch Critic",
        "You are a comedy critic. Evaluate the jokes and respond ONLY with valid JSON "
        "in this exact format (no markdown, no extra text):\n"
        '{"originality": <1-10>, "timing": <1-10>, "punchline": <1-10>, '
        '"overall": <1-10>, "one_liner_verdict": "<string>"}'
    )

    results = []
    for topic in topics:
        print(f"  🎯 Processing: '{topic}'...", end=" ", flush=True)
        
        # Get joke
        joke = batch_comedian.chat(f"Tell one short joke about: '{topic}'")
        batch_comedian.reset()  # Fresh for each topic

        # Get scores as JSON
        raw_score = batch_critic.chat(
            f"Topic: '{topic}'\nJoke: {joke}\n\nEvaluate and respond with JSON only."
        )
        batch_critic.reset()

        try:
            # Strip any accidental markdown fences
            clean = re.sub(r"```json|```", "", raw_score).strip()
            scores = json.loads(clean)
            scores["topic"] = topic
            scores["joke"] = joke
            results.append(scores)
            print(f"Overall: {scores['overall']}/10")
        except json.JSONDecodeError:
            print("⚠️  Parse error — skipping")

    return results


topics_list = ["Social Media", "Coffee Addiction", "Working From Home", "Dating Apps", "Climate Change"]

print("🎬 Running batch evaluation...")
results = batch_comedy_evaluation(topics_list)
print("\n✅ Batch complete!")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def visualize_scores(results: list[dict]):
    """Bar chart of comedian performance across topics."""
    if not results:
        print("No results to visualize.")
        return

    topics     = [r["topic"] for r in results]
    orig       = [r.get("originality", 0) for r in results]
    timing     = [r.get("timing", 0) for r in results]
    punchline  = [r.get("punchline", 0) for r in results]
    overall    = [r.get("overall", 0) for r in results]

    x = np.arange(len(topics))
    width = 0.2

    fig, ax = plt.subplots(figsize=(13, 6))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#16213e')

    colors = ['#e94560', '#0f3460', '#533483', '#e8d5b7']
    bars = [
        ax.bar(x - 1.5*width, orig,      width, label='Originality', color=colors[0], alpha=0.9),
        ax.bar(x - 0.5*width, timing,    width, label='Timing',      color=colors[1], alpha=0.9),
        ax.bar(x + 0.5*width, punchline, width, label='Punchline',   color=colors[2], alpha=0.9),
        ax.bar(x + 1.5*width, overall,   width, label='Overall',     color=colors[3], alpha=0.9),
    ]

    ax.set_xlabel('Topic', color='white', fontsize=12)
    ax.set_ylabel('Score (1–10)', color='white', fontsize=12)
    ax.set_title('🎤 Multi-Agent Comedy Scores by Topic', color='white', fontsize=15, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(topics, rotation=20, ha='right', color='white')
    ax.set_ylim(0, 10.5)
    ax.tick_params(colors='white')
    ax.spines[['bottom', 'left']].set_color('grey')
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=10)
    ax.yaxis.grid(True, color='grey', linestyle='--', alpha=0.3)

    plt.tight_layout()
    plt.savefig("/mnt/user-data/outputs/comedy_scores.png", dpi=120, bbox_inches='tight')
    plt.show()
    print("📊 Chart saved!")


visualize_scores(results)

## 📋 Step 9: Print Score Summary Table

In [ ]:
def print_summary_table(results: list[dict]):
    """Prints a clean summary table of all scores."""
    header = f"{'Topic':<22} {'Original':>9} {'Timing':>8} {'Punchline':>11} {'Overall':>9}"
    print("\n" + "=" * 63)
    print("  📊 COMEDY PERFORMANCE SUMMARY")
    print("=" * 63)
    print(header)
    print("-" * 63)
    
    for r in results:
        row = (f"{r['topic']:<22} "
               f"{r.get('originality', 'N/A'):>9} "
               f"{r.get('timing', 'N/A'):>8} "
               f"{r.get('punchline', 'N/A'):>11} "
               f"{r.get('overall', 'N/A'):>9}")
        print(row)
    
    # Averages
    avg = lambda key: round(sum(r.get(key, 0) for r in results) / len(results), 1)
    print("-" * 63)
    print(f"{'AVERAGE':<22} {avg('originality'):>9} {avg('timing'):>8} "
          f"{avg('punchline'):>11} {avg('overall'):>9}")
    print("=" * 63)

    # Verdicts
    print("\n  📝 CRITIC'S ONE-LINERS:")
    for r in results:
        print(f"  • {r['topic']}: {r.get('one_liner_verdict', 'N/A')}")


print_summary_table(results)

## 🎓 Step 10: Key Concepts Recap

```
Multi-Agent Architecture Patterns Used:
┌─────────────────────────────────────────────────────────┐
│  1. ORCHESTRATOR PATTERN                                │
│     MC agent controls the flow and order of agents      │
│                                                         │
│  2. PIPELINE PATTERN                                    │
│     Setup → Punchline → Polisher (sequential handoff)   │
│                                                         │
│  3. DEBATE PATTERN                                      │
│     Two agents compete; a judge evaluates each round    │
│                                                         │
│  4. CRITIC-IN-THE-LOOP                                  │
│     Agent output is evaluated → feedback → revision     │
│                                                         │
│  5. STRUCTURED OUTPUT PARSING                           │
│     Critic returns JSON → parsed for visualization      │
└─────────────────────────────────────────────────────────┘
```

### Extensions to Try
- Add a **MemoryAgent** that learns which joke styles score highest
- Use **function calling** so the critic returns structured scores natively
- Add **async** API calls for parallel multi-agent execution
- Swap in `gpt-4o-mini` for the faster/cheaper heckler and polisher agents
- Build a **Streamlit or Gradio** front-end around this pipeline